In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

GOLD_CATALOG = "main"
GOLD_SCHEMA  = "gold"
SILVER_CORE  = "main.silver.cms_beneficiary_core"
SILVER_MONTH = "main.silver.cms_beneficiary_monthly"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_CATALOG}.{GOLD_SCHEMA}")
print("Setup complete.")

In [0]:
core    = spark.table(SILVER_CORE)
monthly = spark.table(SILVER_MONTH)

print(f"core    : {core.count():,} rows")
print(f"monthly : {monthly.count():,} rows")

#  Lookup Dictionaries (Label Mapping)

In [0]:
SEX_MAP = {1: "Male", 2: "Female", 0: "Unknown"}

RACE_MAP = {
    1: "White",
    2: "Black",
    3: "Other",
    4: "Asian/Pacific Islander",
    5: "Hispanic",
    6: "American Indian/Alaska Native",
}

DUAL_MAP = {
    "00": "Not dual",
    "01": "Full dual",
    "02": "Partial dual",
    "03": "Full dual (QMB)",
    "04": "Partial dual (QMB)",
    "05": "Partial dual (SLMB)",
    "06": "Partial dual (QI)",
    "08": "Other partial dual",
    "09": "Other full dual",
    "NA": "Not applicable",
}

def label_column(df, col_name, mapping, out_name):
    """Map coded column to labels; unknown codes pass through as strings."""
    expr = F.col(col_name).cast("string")
    for k, v in mapping.items():
        expr = F.when(F.col(col_name).cast("string") == str(k), v).otherwise(expr)
    return df.withColumn(out_name, expr)

# gold enrollment by year

In [0]:
g_enroll_year = (core
    .groupBy("_ref_year")
    .agg(
        F.countDistinct("BENE_ID").alias("beneficiaries"),
        F.round(F.avg("AGE_AT_END_REF_YR"), 1).alias("avg_age"),
        F.sum(F.when(F.col("is_deceased"), 1).otherwise(0)).alias("deceased_count"),
        F.sum(F.when(F.col("ESRD_IND") == "Y", 1).otherwise(0)).alias("esrd_count"),
    )
    .withColumn("deceased_pct",
                F.round(100.0 * F.col("deceased_count") / F.col("beneficiaries"), 2))
    .withColumn("esrd_pct",
                F.round(100.0 * F.col("esrd_count") / F.col("beneficiaries"), 2))
    .orderBy("_ref_year"))

(g_enroll_year.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_enrollment_by_year"))

display(g_enroll_year)

# gold enrollment by state year

In [0]:
g_state_year = (core
    .groupBy("_ref_year", "STATE_CODE")
    .agg(
        F.countDistinct("BENE_ID").alias("beneficiaries"),
        F.round(F.avg("AGE_AT_END_REF_YR"), 1).alias("avg_age"),
        F.sum(F.when(F.col("is_deceased"), 1).otherwise(0)).alias("deceased_count"),
    )
    .withColumn("deceased_pct",
                F.round(100.0 * F.col("deceased_count") / F.col("beneficiaries"), 2))
    .orderBy("_ref_year", F.desc("beneficiaries")))

(g_state_year.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ref_year")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_enrollment_by_state_year"))

print(f"✓ {g_state_year.count():,} rows")

# gold demographics by year

In [0]:
g_demographics = (core
    .transform(lambda d: label_column(d, "SEX_IDENT_CD",  SEX_MAP,  "sex"))
    .transform(lambda d: label_column(d, "BENE_RACE_CD",  RACE_MAP, "race"))
    .groupBy("_ref_year", "age_band", "sex", "race")
    .agg(F.countDistinct("BENE_ID").alias("beneficiaries"))
)

# Add % of year total
year_totals = (g_demographics
    .groupBy("_ref_year")
    .agg(F.sum("beneficiaries").alias("year_total")))

g_demographics = (g_demographics
    .join(year_totals, on="_ref_year")
    .withColumn("pct_of_year",
                F.round(100.0 * F.col("beneficiaries") / F.col("year_total"), 2))
    .drop("year_total"))

(g_demographics.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ref_year")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_demographics_by_year"))

print(f"✓ {g_demographics.count():,} rows")

# gold dual eligible share

In [0]:
g_dual = (monthly
    .filter(F.col("dual_status_code").isNotNull())
    .groupBy("_ref_year", "BENE_ID")
    .agg(F.max(
        F.when(
            (F.col("dual_status_code") != "NA") &
            (F.col("dual_status_code") != "00"),
            1
        ).otherwise(0)
    ).alias("is_dual"))
    .groupBy("_ref_year")
    .agg(
        F.countDistinct("BENE_ID").alias("total_benes"),
        F.sum("is_dual").alias("dual_benes"),
    )
    .withColumn("dual_share_pct",
                F.round(100.0 * F.col("dual_benes") / F.col("total_benes"), 2))
    .orderBy("_ref_year"))

(g_dual.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_dual_eligible_share"))

display(g_dual)

# gold Medicare Advantage  vs Original Medicare  penetration

In [0]:
g_ma_ffs = (monthly
    .groupBy("_ref_year", "BENE_ID")
    .agg(
        # MA if bene has any PTC contract in any month
        F.max(
            F.when(F.col("ptc_contract_id").isNotNull(), 1).otherwise(0)
        ).alias("is_ma"),
        # HMO proxy
        F.max(
            F.when(F.col("hmo_ind") == "Y", 1).otherwise(0)
        ).alias("is_hmo"),
        # Part D (PDP/MA-PD) if any PTD contract
        F.max(
            F.when(F.col("ptd_contract_id").isNotNull(), 1).otherwise(0)
        ).alias("has_partd"),
    )
    .groupBy("_ref_year")
    .agg(
        F.countDistinct("BENE_ID").alias("total_benes"),
        F.sum("is_ma").alias("ma_benes"),
        F.sum("is_hmo").alias("hmo_benes"),
        F.sum("has_partd").alias("partd_benes"),
    )
    .withColumn("ma_share_pct",
                F.round(100.0 * F.col("ma_benes") / F.col("total_benes"), 2))
    .withColumn("hmo_share_pct",
                F.round(100.0 * F.col("hmo_benes") / F.col("total_benes"), 2))
    .withColumn("partd_share_pct",
                F.round(100.0 * F.col("partd_benes") / F.col("total_benes"), 2))
    .orderBy("_ref_year"))

(g_ma_ffs.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_ma_vs_ffs_penetration"))

display(g_ma_ffs)

# gold plan contracts

In [0]:
g_contracts = (monthly
    .filter(F.col("ptc_contract_id").isNotNull())
    .groupBy("_ref_year", "ptc_contract_id", "ptc_plan_type")
    .agg(F.countDistinct("BENE_ID").alias("beneficiaries"))
    .orderBy("_ref_year", F.desc("beneficiaries")))

(g_contracts.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ref_year")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_plan_contracts"))

print(f"✓ {g_contracts.count():,} rows")

# gold deceased cohort

In [0]:
g_deceased = (core
    .groupBy("_ref_year", "age_band")
    .agg(
        F.countDistinct("BENE_ID").alias("total_benes"),
        F.sum(F.when(F.col("is_deceased"), 1).otherwise(0)).alias("deceased"),
    )
    .withColumn("mortality_pct",
                F.round(100.0 * F.col("deceased") / F.col("total_benes"), 3))
    .orderBy("_ref_year", "age_band"))

(g_deceased.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_deceased_cohort"))

display(g_deceased)

# gold state year kpis

In [0]:
# ============================================================
# GOLD — gold_state_year_kpis
# ============================================================
from pyspark.sql import functions as F

# ------------------------------------------------------------
# 1) Bring STATE_CODE into monthly via a join with core
# ------------------------------------------------------------
# core already has STATE_CODE, BENE_ID, _ref_year → use as the state lookup
state_lookup = core.select("BENE_ID", "_ref_year", "STATE_CODE").distinct()

# monthly already has (BENE_ID, _ref_year, month_num, dual_status_code, ptc_contract_id)
monthly_with_state = (monthly
    .join(state_lookup, on=["BENE_ID", "_ref_year"], how="left"))

# Sanity: make sure the join worked
missing_state = monthly_with_state.filter(F.col("STATE_CODE").isNull()).count()
print(f"monthly rows missing STATE_CODE after join: {missing_state:,}")

# ------------------------------------------------------------
# 2) State base metrics (from core)
# ------------------------------------------------------------
state_base = (core
    .groupBy("_ref_year", "STATE_CODE")
    .agg(
        F.countDistinct("BENE_ID").alias("total_benes"),
        F.round(F.avg("AGE_AT_END_REF_YR"), 1).alias("avg_age"),
        F.sum(F.when(F.col("is_deceased"), 1).otherwise(0)).alias("deceased"),
        F.sum(F.when(F.col("ESRD_IND") == "Y", 1).otherwise(0)).alias("esrd"),
    ))

# ------------------------------------------------------------
# 3) Dual-eligible counts by state/year
# ------------------------------------------------------------
dual_state = (monthly_with_state
    .filter(F.col("dual_status_code").isNotNull())
    .groupBy("_ref_year", "STATE_CODE", "BENE_ID")
    .agg(F.max(
        F.when(
            (F.col("dual_status_code") != "NA") &
            (F.col("dual_status_code") != "00"),
            1
        ).otherwise(0)
    ).alias("is_dual"))
    .groupBy("_ref_year", "STATE_CODE")
    .agg(F.sum("is_dual").alias("dual_benes")))

# ------------------------------------------------------------
# 4) MA counts by state/year
# ------------------------------------------------------------
ma_state = (monthly_with_state
    .groupBy("_ref_year", "STATE_CODE", "BENE_ID")
    .agg(F.max(
        F.when(F.col("ptc_contract_id").isNotNull(), 1).otherwise(0)
    ).alias("is_ma"))
    .groupBy("_ref_year", "STATE_CODE")
    .agg(F.sum("is_ma").alias("ma_benes")))

# ------------------------------------------------------------
# 5) Join + compute KPIs
# ------------------------------------------------------------
g_state_kpis = (state_base
    .join(dual_state, on=["_ref_year", "STATE_CODE"], how="left")
    .join(ma_state,   on=["_ref_year", "STATE_CODE"], how="left")
    .fillna(0, subset=["dual_benes", "ma_benes"])
    .withColumn("dual_share_pct",
                F.round(100.0 * F.col("dual_benes") / F.col("total_benes"), 2))
    .withColumn("ma_share_pct",
                F.round(100.0 * F.col("ma_benes")   / F.col("total_benes"), 2))
    .withColumn("deceased_pct",
                F.round(100.0 * F.col("deceased")   / F.col("total_benes"), 3))
    .withColumn("esrd_pct",
                F.round(100.0 * F.col("esrd")       / F.col("total_benes"), 3))
    .orderBy("_ref_year", F.desc("total_benes")))

# ------------------------------------------------------------
# 6) Persist
# ------------------------------------------------------------
(g_state_kpis.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ref_year")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_state_year_kpis"))

print(f"✓ gold_state_year_kpis rows: {g_state_kpis.count():,}")
display(g_state_kpis)

In [0]:
%sql
-- Every gold table, row counts
SELECT 'gold_enrollment_by_year'      AS table_name, COUNT(*) AS rows FROM main.gold.gold_enrollment_by_year
UNION ALL SELECT 'gold_enrollment_by_state_year', COUNT(*) FROM main.gold.gold_enrollment_by_state_year
UNION ALL SELECT 'gold_demographics_by_year',     COUNT(*) FROM main.gold.gold_demographics_by_year
UNION ALL SELECT 'gold_dual_eligible_share',      COUNT(*) FROM main.gold.gold_dual_eligible_share
UNION ALL SELECT 'gold_ma_vs_ffs_penetration',    COUNT(*) FROM main.gold.gold_ma_vs_ffs_penetration
UNION ALL SELECT 'gold_plan_contracts',           COUNT(*) FROM main.gold.gold_plan_contracts
UNION ALL SELECT 'gold_deceased_cohort',          COUNT(*) FROM main.gold.gold_deceased_cohort
UNION ALL SELECT 'gold_state_year_kpis',          COUNT(*) FROM main.gold.gold_state_year_kpis
ORDER BY table_name;

In [0]:
%sql
-- Enrollment trend
SELECT * FROM main.gold.gold_enrollment_by_year ORDER BY _ref_year;

In [0]:
%sql
-- MA penetration over time
SELECT _ref_year, ma_share_pct, hmo_share_pct, partd_share_pct
FROM   main.gold.gold_ma_vs_ffs_penetration
ORDER BY _ref_year;

In [0]:
%sql
-- Top 10 states by bene count in 2025
SELECT STATE_CODE, total_benes, avg_age, dual_share_pct, ma_share_pct
FROM   main.gold.gold_state_year_kpis
WHERE  _ref_year = 2025
ORDER BY total_benes DESC
LIMIT 10;

In [0]:
latest_year = core.agg(F.max("_ref_year")).collect()[0][0]

g_latest = (spark.table(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_state_year_kpis")
    .filter(F.col("_ref_year") == latest_year))

(g_latest.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_CATALOG}.{GOLD_SCHEMA}.gold_state_kpis_latest"))

print(f"✓ Latest snapshot for {latest_year}: {g_latest.count():,} rows")